# ARES 2023-68A — Fill Unpriced Loans

Six loans in the derived LLD have no `Last Price`. We fill each one **directly into
the `Last Price` column** using a **tiered** method, record which tier in
`price_source`, and note how much the fill moved the portfolio mark.

**Tiered method (recorded per loan in `price_source`):**
1. **`transacted_monthly`** — a recent transacted price for the line from the
   Monthly Report purchases/sales section (match issuer/CUSIP).
2. **`rating_cohort_wa`** *(primary fallback)* — the **par-weighted** average
   `Last Price` of the **priced, non-defaulted** loans in the **same S&P Reported
   Rating cohort**. *Not* the flat portfolio WA — flat is ~95, but the CCC+
   cohort trades ~63, so a flat mark would be ~30 pts rich on that paper.

**Tier-1 search result (this deal):** the Monthly Report (`Monthly_Report_05_26.pdf`)
is a *Principal Activity Report* — it lists transaction **cashflow amounts**, not
per-unit prices, so no clean transacted price is recoverable for these lines.
Clarios appears with **interest payments only** (no fresh trade) → consistent with
an identifier-mapping issue, not a true purchase. So all six fall to tier 2.
(A `TRANSACTED` override dict is provided below if a clean price is later found.)

The filled values are written into `Last Price` (originally-priced rows are
unchanged). `price_source` distinguishes real Bloomberg marks (`reported`) from the
6 estimates (`rating_cohort_wa`), so the fill stays auditable. Raw `.xlsx` untouched.

In [1]:
import os, glob
import numpy as np, pandas as pd

DERIVED = next(p for p in ['output/lld_ares_2023_derived.csv',
                           '../output/lld_ares_2023_derived.csv',
                           'lld_ares_2023_derived.csv'] if os.path.exists(p))
df = pd.read_csv(DERIVED)
df = df.drop(columns=[c for c in ['price_filled'] if c in df.columns])  # drop any stale fill col
print('LLD:', df.shape)

price   = pd.to_numeric(df['Last Price'], errors='coerce')
bal     = df['Current Balance']
non_def = df['Default Status'].astype(str).str.lower() != 'yes'

# Identify the originally-REPORTED rows. We fill Last Price in place, so on a re-run
# the blanks are gone; to stay idempotent we trust a persisted `price_source` if the
# file already has one, and only fall back to "which Last Prices are present" on the
# first pass. This keeps the 6 fills flagged no matter how many times we re-run.
if 'price_source' in df.columns:
    reported_mask = df['price_source'].astype(str) == 'reported'
else:
    reported_mask = price.notna() & (price > 0)
unpriced_mask = ~reported_mask

unpriced = df.loc[unpriced_mask, ['Issuer','CUSIP','S&P Reported Rating','Current Balance']]
print('originally-unpriced loans:', int(unpriced_mask.sum()))
print(unpriced.to_string())

LLD: (415, 55)
originally-unpriced loans: 6
                         Issuer      CUSIP S&P Reported Rating  Current Balance
36             BW Holding, Inc.  12430EAD6                CCC+       2842990.02
106   WideOpenWest Finance, LLC  96758DBL6                  B-       1727906.98
204   Versant Media Group, Inc.        NaN                  BB        900000.00
249           Clarios Global LP  C8000CAP8                 BB-        638696.62
250   DEEP BLUE OPERATING I LLC  24369TAD3                 BB-        635000.00
297  Pretium PKG Holdings, Inc.  74142KAQ2                CCC+        462592.80


In [2]:
# --- Tier 1: transacted price from Monthly Report (manual overrides) ---
# key = row index; value = transacted price. Empty: no clean per-unit price found
# in the Principal Activity Report (cashflow amounts only; Clarios = interest-only,
# an identifier-mapping issue, not a fresh trade).
TRANSACTED = {}   # e.g. {249: 99.5}

# reset provenance from the reported-mask, then apply any transacted overrides
df['price_source'] = np.where(reported_mask, 'reported', 'unpriced')
for idx, px in TRANSACTED.items():
    df.at[idx, 'Last Price'] = px
    df.at[idx, 'price_source'] = 'transacted_monthly'
print('tier-1 transacted fills:', len(TRANSACTED))

tier-1 transacted fills: 0


/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_90122/2285945365.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['price_source'] = np.where(reported_mask, 'reported', 'unpriced')


In [3]:
# --- Tier 2: rating-cohort par-weighted WA (reported, non-defaulted) ---
# cohort uses ONLY the originally-reported real marks (reported_mask), so re-runs
# never contaminate the cohort with previously-filled estimates.
elig = reported_mask & non_def
cohort_wa = (df[elig].groupby('S&P Reported Rating')
             .apply(lambda g: np.average(price[g.index], weights=bal[g.index])))
flat_wa = np.average(price[elig], weights=bal[elig])   # only a documented last-resort
print('flat portfolio WA (non-def): %.2f'% flat_wa)
print('cohort WAs:'); print(cohort_wa.round(2).to_string())

to_fill = df['price_source'] == 'unpriced'
for idx in df.index[to_fill]:
    r = df.at[idx, 'S&P Reported Rating']
    df.at[idx, 'Last Price'] = cohort_wa.get(r, flat_wa)
    df.at[idx, 'price_source'] = 'rating_cohort_wa' if r in cohort_wa.index else 'flat_wa_fallback'

print('\nfilled (now written into Last Price):')
print(df.loc[unpriced_mask, ['Issuer','S&P Reported Rating','Last Price','price_source']]
      .round(2).to_string())

flat portfolio WA (non-def): 95.36
cohort WAs:
S&P Reported Rating
A        93.80
A+       88.81
A-       92.39
AA-      89.40
B        98.34
B+       98.32
B-       95.66
BB       99.81
BB+     100.02
BB-      99.92
BBB     100.58
BBB+     96.44
BBB-     99.86
CCC      65.36
CCC+     63.41
CCC-     37.92

filled (now written into Last Price):
                         Issuer S&P Reported Rating  Last Price      price_source
36             BW Holding, Inc.                CCC+       63.41  rating_cohort_wa
106   WideOpenWest Finance, LLC                  B-       95.66  rating_cohort_wa
204   Versant Media Group, Inc.                  BB       99.81  rating_cohort_wa
249           Clarios Global LP                 BB-       99.92  rating_cohort_wa
250   DEEP BLUE OPERATING I LLC                 BB-       99.92  rating_cohort_wa
297  Pretium PKG Holdings, Inc.                CCC+       63.41  rating_cohort_wa


/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_90122/791321383.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(price[g.index], weights=bal[g.index])))


In [4]:
# --- Save tape with Last Price filled in place (raw .xlsx untouched) ---
OUT = DERIVED   # overwrite the derived tape; Last Price now has no blanks
df.to_csv(OUT, index=False)
print('wrote', os.path.abspath(OUT))
print('Last Price nulls remaining:', df['Last Price'].isna().sum())
print('price_source counts:'); print(df['price_source'].value_counts().to_string())

wrote /Users/amine/Documents/Columbia/Classes/ENGIE 4700 - Summer Project/correlation_and_tail_risk_in_clo_tranches/project/notebooks/output/lld_ares_2023_derived.csv
Last Price nulls remaining: 0
price_source counts:
price_source
reported            409
rating_cohort_wa      6


In [5]:
# --- Note: how much did the fill move the portfolio mark? ---
# Compare cohort fills vs a naive flat-WA fill for the originally-unpriced loans.
flat_fill = price.copy()
flat_fill[unpriced_mask] = flat_wa
port_cohort = np.average(df['Last Price'], weights=bal)   # Last Price now carries the cohort fills
port_flat   = np.average(flat_fill,        weights=bal)
move_pts = port_cohort - port_flat
move_pct = move_pts / port_flat * 100

note = (f"Portfolio WA price (cohort fills): {port_cohort:.3f}\n"
        f"Portfolio WA price (naive flat fills): {port_flat:.3f}\n"
        f"Move from using rating-cohort marks: {move_pts:+.3f} pts ({move_pct:+.2f}%)\n"
        f"Driver: 2 CCC+ names (BW Holding, Pretium PKG) marked at the CCC+ cohort "
        f"~{cohort_wa.get('CCC+', float('nan')):.1f} instead of the flat ~{flat_wa:.1f} "
        f"(~{flat_wa - cohort_wa.get('CCC+', flat_wa):.0f} pts richer if flat-marked).")
print(note)
with open(os.path.join(os.path.dirname(DERIVED), 'pricing_move_note.txt'), 'w') as f:
    f.write(note + '\n')

Portfolio WA price (cohort fills): 94.884
Portfolio WA price (naive flat fills): 95.078
Move from using rating-cohort marks: -0.194 pts (-0.20%)
Driver: 2 CCC+ names (BW Holding, Pretium PKG) marked at the CCC+ cohort ~63.4 instead of the flat ~95.4 (~32 pts richer if flat-marked).
